In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 【读取步骤】
# 读取测试集预测结果和清洗后的测试集特征
# ============================================================

test_predictions = pd.read_csv(
    "data/processed/lightgbm_test_predictions.csv"
)

model_test = pd.read_parquet(
    "data/processed/model_test_clean.parquet"
)


print(
    "预测结果 Shape:",
    test_predictions.shape
)

print(
    "测试集特征 Shape:",
    model_test.shape
)

print(
    "预测结果重复客户数量:",
    test_predictions["SK_ID_CURR"].duplicated().sum()
)

print(
    "测试集重复客户数量:",
    model_test["SK_ID_CURR"].duplicated().sum()
)

预测结果 Shape: (48744, 2)
测试集特征 Shape: (48744, 440)
预测结果重复客户数量: 0
测试集重复客户数量: 0


In [2]:
# ============================================================
# 【合并步骤】
# 将预测违约概率合并到测试集客户特征
# ============================================================

risk_portfolio = (
    model_test
    .merge(
        test_predictions.rename(
            columns={
                "TARGET": "PREDICTED_DEFAULT_PROBABILITY"
            }
        ),
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)


print(
    "风险组合数据 Shape:",
    risk_portfolio.shape
)

print(
    "预测概率缺失数量:",
    risk_portfolio[
        "PREDICTED_DEFAULT_PROBABILITY"
    ].isna().sum()
)

print(
    "重复客户数量:",
    risk_portfolio[
        "SK_ID_CURR"
    ].duplicated().sum()
)

风险组合数据 Shape: (48744, 441)
预测概率缺失数量: 0
重复客户数量: 0


In [3]:
# ============================================================
# 【特征工程 1】
# 根据预测违约概率建立固定风险等级
# ============================================================

risk_portfolio["RISK_LEVEL"] = pd.cut(
    risk_portfolio[
        "PREDICTED_DEFAULT_PROBABILITY"
    ],
    bins=[
        -np.inf,
        0.10,
        0.30,
        0.50,
        np.inf
    ],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk",
        "Very High Risk"
    ],
    right=False
)


# 2. 建立风险等级排序字段
risk_level_order = {
    "Low Risk": 1,
    "Medium Risk": 2,
    "High Risk": 3,
    "Very High Risk": 4
}

risk_portfolio["RISK_LEVEL_ORDER"] = (
    risk_portfolio["RISK_LEVEL"]
    .astype("string")
    .map(risk_level_order)
    .astype("int8")
)


# 3. 查看风险等级分布
risk_portfolio[
    "RISK_LEVEL"
].value_counts(
    sort=False
)

RISK_LEVEL
Low Risk           4097
Medium Risk       18671
High Risk         12754
Very High Risk    13222
Name: count, dtype: int64

In [4]:
# ============================================================
# 【特征工程 2】
# 建立0至100风险评分
# ============================================================

risk_portfolio["RISK_SCORE"] = (
    risk_portfolio[
        "PREDICTED_DEFAULT_PROBABILITY"
    ]
    * 100
).round(2)


risk_portfolio[
    [
        "SK_ID_CURR",
        "PREDICTED_DEFAULT_PROBABILITY",
        "RISK_SCORE",
        "RISK_LEVEL"
    ]
].head()

,SK_ID_CURR,PREDICTED_DEFAULT_PROBABILITY,RISK_SCORE,RISK_LEVEL
0,100001,0.307499,30.75,High Risk
1,100005,0.643616,64.36,Very High Risk
2,100013,0.245754,24.58,Medium Risk
3,100028,0.171460,17.15,Medium Risk
4,100038,0.653619,65.36,Very High Risk


In [5]:
# ============================================================
# 【汇总步骤 1】
# 汇总不同风险等级的客户规模
# ============================================================

risk_level_summary = (
    risk_portfolio
    .groupby(
        "RISK_LEVEL",
        observed=False
    )
    .agg(
        CUSTOMER_COUNT=(
            "SK_ID_CURR",
            "nunique"
        ),

        AVG_DEFAULT_PROBABILITY=(
            "PREDICTED_DEFAULT_PROBABILITY",
            "mean"
        ),

        MIN_DEFAULT_PROBABILITY=(
            "PREDICTED_DEFAULT_PROBABILITY",
            "min"
        ),

        MAX_DEFAULT_PROBABILITY=(
            "PREDICTED_DEFAULT_PROBABILITY",
            "max"
        ),

        AVG_RISK_SCORE=(
            "RISK_SCORE",
            "mean"
        )
    )
    .reset_index()
)


risk_level_summary[
    "CUSTOMER_RATE"
] = (
    risk_level_summary[
        "CUSTOMER_COUNT"
    ]
    / risk_level_summary[
        "CUSTOMER_COUNT"
    ].sum()
)


risk_level_summary

,RISK_LEVEL,CUSTOMER_COUNT,AVG_DEFAULT_PROBABILITY,MIN_DEFAULT_PROBABILITY,MAX_DEFAULT_PROBABILITY,AVG_RISK_SCORE,CUSTOMER_RATE
0,Low Risk,4097,0.073600,0.018304,0.099996,7.360056,0.084051
1,Medium Risk,18671,0.195413,0.100006,0.299985,19.541320,0.383042
2,High Risk,12754,0.393343,0.300010,0.499997,39.334385,0.261653
3,Very High Risk,13222,0.655668,0.500031,0.946449,65.566743,0.271254


In [6]:
# ============================================================
# 【汇总步骤 2】
# 汇总不同风险等级的贷款和收入表现
# ============================================================

risk_financial_summary = (
    risk_portfolio
    .groupby(
        "RISK_LEVEL",
        observed=False
    )
    .agg(
        CUSTOMER_COUNT=(
            "SK_ID_CURR",
            "nunique"
        ),

        TOTAL_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "sum"
        ),

        AVG_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "mean"
        ),

        AVG_INCOME_AMOUNT=(
            "AMT_INCOME_TOTAL",
            "mean"
        ),

        AVG_ANNUITY_AMOUNT=(
            "AMT_ANNUITY",
            "mean"
        ),

        AVG_LOAN_TO_INCOME=(
            "LOAN_TO_INCOME",
            "mean"
        ),

        AVG_ANNUITY_TO_INCOME=(
            "ANNUITY_TO_INCOME",
            "mean"
        )
    )
    .reset_index()
)


risk_financial_summary

,RISK_LEVEL,CUSTOMER_COUNT,TOTAL_CREDIT_AMOUNT,AVG_CREDIT_AMOUNT,AVG_INCOME_AMOUNT,AVG_ANNUITY_AMOUNT,AVG_LOAN_TO_INCOME,AVG_ANNUITY_TO_INCOME
0,Low Risk,4097,2.098196e+09,512129.894191,192564.054918,29785.629607,2.767929,0.166598
1,Medium Risk,18671,1.038242e+10,556072.047534,184503.538589,29953.724707,3.272090,0.179924
2,High Risk,12754,6.533584e+09,512277.238396,174400.026979,28990.618081,3.247512,0.184746
3,Very High Risk,13222,6.173795e+09,466933.483323,169367.833800,28982.002231,3.066602,0.190646


In [7]:
# ============================================================
# 【汇总步骤 3】
# 汇总不同风险等级的历史信用表现
# ============================================================

risk_history_summary = (
    risk_portfolio
    .groupby(
        "RISK_LEVEL",
        observed=False
    )
    .agg(
        CUSTOMER_COUNT=(
            "SK_ID_CURR",
            "nunique"
        ),

        BUREAU_HISTORY_RATE=(
            "HAS_BUREAU_HISTORY",
            "mean"
        ),

        PREVIOUS_APPLICATION_HISTORY_RATE=(
            "HAS_PREVIOUS_APPLICATION_HISTORY",
            "mean"
        ),

        INSTALLMENT_HISTORY_RATE=(
            "HAS_INSTALLMENT_HISTORY",
            "mean"
        ),

        POS_CASH_HISTORY_RATE=(
            "HAS_POS_CASH_HISTORY",
            "mean"
        ),

        CREDIT_CARD_HISTORY_RATE=(
            "HAS_CREDIT_CARD_HISTORY",
            "mean"
        ),

        AVG_PREVIOUS_REFUSED_RATE=(
            "PREV_REFUSED_RATE",
            "mean"
        ),

        AVG_INSTALLMENT_LATE_RATE=(
            "LATE_INSTALMENT_RATE",
            "mean"
        ),

        AVG_POS_DPD_RATE=(
            "POS_TOTAL_DPD_MONTH_RATE",
            "mean"
        ),

        AVG_CREDIT_CARD_DPD_RATE=(
            "CC_TOTAL_DPD_MONTH_RATE",
            "mean"
        )
    )
    .reset_index()
)


risk_history_summary

,RISK_LEVEL,CUSTOMER_COUNT,BUREAU_HISTORY_RATE,PREVIOUS_APPLICATION_HISTORY_RATE,INSTALLMENT_HISTORY_RATE,POS_CASH_HISTORY_RATE,CREDIT_CARD_HISTORY_RATE,AVG_PREVIOUS_REFUSED_RATE,AVG_INSTALLMENT_LATE_RATE,AVG_POS_DPD_RATE,AVG_CREDIT_CARD_DPD_RATE
0,Low Risk,4097,0.959238,0.980474,0.981938,0.979985,0.370271,0.053772,0.025867,0.006828,0.003943
1,Medium Risk,18671,0.902898,0.979380,0.983718,0.979808,0.346045,0.086337,0.056284,0.013925,0.005396
2,High Risk,12754,0.847420,0.980869,0.983927,0.981574,0.339188,0.120904,0.080977,0.018346,0.006146
3,Very High Risk,13222,0.811072,0.982227,0.983588,0.981697,0.328921,0.159013,0.103751,0.023579,0.005471


In [8]:
# ============================================================
# 【检查步骤】
# 验证风险等级是否正确
# ============================================================

risk_level_validation = (
    risk_portfolio
    .groupby(
        "RISK_LEVEL",
        observed=False
    )[
        "PREDICTED_DEFAULT_PROBABILITY"
    ]
    .agg(
        [
            "count",
            "min",
            "max",
            "mean"
        ]
    )
)


risk_level_validation

,count,min,max,mean
RISK_LEVEL,,,,
Low Risk,4097,0.018304,0.099996,0.073600
Medium Risk,18671,0.100006,0.299985,0.195413
High Risk,12754,0.300010,0.499997,0.393343
Very High Risk,13222,0.500031,0.946449,0.655668


In [9]:
# ============================================================
# 【保存结果】
# 保存客户级风险结果和风险组合摘要
# ============================================================

output_dir = Path(
    "data/processed"
)


# 1. 保存客户级风险评分表
risk_portfolio.to_parquet(
    output_dir / "customer_risk_portfolio.parquet",
    index=False
)

risk_portfolio.to_csv(
    output_dir / "customer_risk_portfolio.csv",
    index=False
)


# 2. 保存风险等级摘要
risk_level_summary.to_csv(
    output_dir / "risk_level_summary.csv",
    index=False
)


# 3. 保存财务风险摘要
risk_financial_summary.to_csv(
    output_dir / "risk_financial_summary.csv",
    index=False
)


# 4. 保存历史行为摘要
risk_history_summary.to_csv(
    output_dir / "risk_history_summary.csv",
    index=False
)


print(
    "风险组合结果保存完成"
)

print(
    "customer_risk_portfolio Shape:",
    risk_portfolio.shape
)

风险组合结果保存完成
customer_risk_portfolio Shape: (48744, 444)
